
# <font color="green">Conditional branch (dereference or zero)</font>

## Problem

* Write a function `deref_or_zero(p)` __in assembly__ that returns `*p` if `p` is non-null, and `0` otherwise.
* The point of this problem is that you **must** use a conditional branch: you cannot load `*p` before checking `p`, because loading from a null pointer would crash.
* That is, translate the following C function into assembly:
```
long deref_or_zero(long * p) {
  if (p) { return *p; } else { return 0; }
}
```
* Compare `p` against `0`, branch to a "return 0" path when it is null, and otherwise load `*p` and return it.
* Fill in the skeleton `deref_or_zero.s` (after `// ------- write your answer here -------`).
* The checker `check_deref_or_zero.c` calls your function with both a valid pointer and a null pointer. If you see `OK`s and no errors, you are done.

## Hints

* In the *Conditional select* problem, the compiler avoided a branch by computing **both** results and selecting one with `csel`. That works only when both sides are always safe and cheap to compute.
* Sometimes that is impossible: only **one** side may be executed. Then the compiler has no choice but to emit a real conditional branch. Typical reasons:
  * executing the wrong side would be **unsafe** --- e.g. dereferencing a pointer that may be null would crash;
  * a side has **side effects** or unknown cost --- e.g. it calls a function, so it cannot be run speculatively.
* The *Observe* cells below contain two such functions: `mod_or_one` (which must not compute `x % y` when `y` is `0`, since dividing by zero is undefined) and `call_or_zero` (which must not call a function speculatively). Compile them and confirm that, unlike `add_or_mul_long`, they really do produce a comparison followed by a conditional branch (e.g. `cbz`/`cbnz` or `cmp` + `b.<cond>`) --- the compiler does **not** compute both sides.



# 1. AI Tutor
## 1-1. Prepare
* Your personal AI tutor is provided for questions and feedback.
* Execute the following cell before you use it.

In [ ]:
import heytutor

## 1-2. Examples
* A general question
```
%%hey
What does the `ldr` instruction do in ARM64?
```

* A hint on this specific problem
```
%%hey problem_file=deref_or_zero.md
Give me a hint on this problem.

{problem}
```

* Builtin variables usable in `%%hey` cells
  * `{file:FILENAME}` is the content of FILE
  * `{bash[-1]}` is the output of the last `%%bash_` cell, `{bash[-2]}` the second last, etc.
  * `{problem}` is the content of the file you specify by `%%hey problem_file=foo.md`
  * `{answer}` is the content of the file you specify by `%%hey answer_file=foo.s`


# 2. Observe: compile example functions
* Before writing your own assembly, it helps to see what the compiler generates for related example functions.
* Running the first cell below writes `explore.c` (some small example functions related to this problem).
* The second cell compiles it with `gcc -O3 -S` and prints the generated assembly.
* Feel free to edit `explore.c` (change the code, add functions, change constants) and re-run the two cells to see how the assembly changes.

In [ ]:
%%writefile_ explore.c
/* These compile to a conditional BRANCH (not a conditional select), because
   only ONE side may safely run --- the compiler must NOT execute both. */

/* It cannot compute x % y speculatively: dividing by zero is undefined,
   so the modulo must stay guarded behind the y != 0 test. */
long mod_or_one(long x, long y) {
  if (y != 0) { return x % y; } else { return 1; }
}

/* It cannot call sink() speculatively: a call may have side effects. */
extern long sink(long x);
long call_or_zero(long x, long c) {
  if (c) { return sink(x); } else { return 0; }
}

In [ ]:
%%bash_
gcc -O3 -S explore.c
cat explore.s


# 3. Your Answer (assembly)
* Running the cell below writes the skeleton assembly file `deref_or_zero.s`.
* Fill in your instructions after the line `// ------- write your answer here -------`, then run the cell again to save it.

In [ ]:
%%writefile_ deref_or_zero.s
	.arch armv8-a
	.file	"deref_or_zero.c"
	.text
	.align	2
	.p2align 4,,11
	.global	deref_or_zero
	.type	deref_or_zero, %function
deref_or_zero:
.LFB0:
	.cfi_startproc
	// ------- write your answer here -------
	.cfi_endproc
.LFE0:
	.size	deref_or_zero, .-deref_or_zero
	.ident	"GCC: (Ubuntu 13.3.0-6ubuntu2~24.04) 13.3.0"
	.section	.note.GNU-stack,"",@progbits


# 4. Checker
* The following C program calls your `deref_or_zero` function and checks the result against a reference computed in C.

In [ ]:
%%writefile_ check_deref_or_zero.c
#include <stdio.h>
long deref_or_zero(long * p);

int main(void) {
  long v = 42;
  long r1 = deref_or_zero(&v);
  long r2 = deref_or_zero((long *) 0);
  int ok = 1;
  if (r1 == 42) printf("OK deref %ld\n", r1);
  else { printf("NG deref %ld (expected 42)\n", r1); ok = 0; }
  if (r2 == 0) printf("OK null %ld\n", r2);
  else { printf("NG null %ld (expected 0)\n", r2); ok = 0; }
  return ok ? 0 : 1;
}


# 5. Compile
* Compile your assembly together with the checker.
* If you get an error, fix `deref_or_zero.s` above and recompile.

In [ ]:
%%bash_
gcc -o check_deref_or_zero -O3 check_deref_or_zero.c deref_or_zero.s -lm


# 6. Run
* If you see `OK`s and no errors, you are done.

In [ ]:
%%bash_
./check_deref_or_zero


# 7. If things do not go well
* If your program compiles but does not produce the correct answer, run it within a debugger (gdb).
* Compile with `-O0 -g` first:
```
gcc -o check_deref_or_zero -O0 -g check_deref_or_zero.c deref_or_zero.s -lm
```
* Then, in a terminal (SSH or Jupyter terminal):
```
gdb check_deref_or_zero
(gdb) break deref_or_zero
(gdb) run ...        # give the same arguments as above
```
* Step through one instruction at a time with `step`, and inspect registers with `print $x0` or `info registers`.

# 8. Ask Questions or Get Feedback
* You are encouraged to ask for feedback once you think you are done, to know if there is a better answer.

In [ ]:
%%hey problem_file=deref_or_zero.md answer_file=deref_or_zero.s

Problem:
{problem}

My Answer:
{answer}

Give me a feedback to my answer.